In [1]:
import pandas as pd
import pickle
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn_quantile import RandomForestQuantileRegressor
from tqdm.auto import tqdm
import CRPS.CRPS as pscore


import multiprocessing as mp
mp.set_start_method('spawn')


import sys
sys.path.append('../../../TaskExecutionTimeMining/')
from quantile_regression import QuantileRegression


sys.path.append('../../../Evaluation/')
import conduct_evaluation
#from normal_evaluation.quantile_regression_evaluation import *
#from normal_evaluation.normal_evaluation import SampleOutcomes_Normal

from PCR_evaluation.pcr_quantile_regression_evaluation import *

get_pscores = lambda likelihoods : [pscore(likelihoods[1][i], likelihoods[2][k][3]).compute()[0] for i, k in enumerate(list(likelihoods[0].keys()))]


In [2]:
with open('./quantile_regression_models.pkl', 'rb') as f:
    quantile_regression_models = pickle.load(f)

In [3]:
with open('../../transformed_event_logs/PCR_start_end_test.pickle', 'rb') as f:
    test_data = pickle.load(f)

activity_count = [
 'Callback timeout',
 'Export result',
 'Export to EMS',
 'Match patient data',
 'Receive sample state',
 'Send notification',
 'Wait for plate validation',
 'timeout',
 ]

ii1 = [
 'intercase_n_1__Callback timeout',
 'intercase_n_1__Export result',
 'intercase_n_1__Export to EMS',
 'intercase_n_1__Match patient data',
 'intercase_n_1__Receive sample state',
 'intercase_n_1__Send notification',
 'intercase_n_1__Wait for plate validation',
 'intercase_n_1__timeout'
]

ii3 = [
 'intercase_n_3__Export result_Callback timeout_Send notification',
 'intercase_n_3__Export result_Export to EMS_Callback timeout',
 'intercase_n_3__Export result_Export to EMS_Send notification',
 'intercase_n_3__Export result_Send notification_Callback timeout',
 'intercase_n_3__Export to EMS_Callback timeout_Send notification',
 'intercase_n_3__Export to EMS_Export result_Callback timeout',
 'intercase_n_3__Export to EMS_Export result_Send notification',
 'intercase_n_3__Export to EMS_Send notification_Callback timeout',
 'intercase_n_3__Match patient data',
 'intercase_n_3__Match patient data_Match patient data',
 'intercase_n_3__Match patient data_Match patient data_Match patient data',
 'intercase_n_3__Match patient data_Match patient data_Receive sample state',
 'intercase_n_3__Match patient data_Match patient data_Send notification',
 'intercase_n_3__Match patient data_Receive sample state_Callback timeout',
 'intercase_n_3__Match patient data_Receive sample state_Export result',
 'intercase_n_3__Match patient data_Receive sample state_Export to EMS',
 'intercase_n_3__Match patient data_Receive sample state_Send notification',
 'intercase_n_3__Match patient data_Send notification_Receive sample state',
 'intercase_n_3__Match patient data_Wait for plate validation',
 'intercase_n_3__Match patient data_Wait for plate validation_Receive sample state',
 'intercase_n_3__Match patient data_Wait for plate validation_Send notification',
 'intercase_n_3__Match patient data_Wait for plate validation_timeout',
 'intercase_n_3__Match patient data_timeout',
 'intercase_n_3__Match patient data_timeout_Match patient data',
 'intercase_n_3__Match patient data_timeout_Receive sample state',
 'intercase_n_3__Match patient data_timeout_Send notification',
 'intercase_n_3__Match patient data_timeout_Wait for plate validation',
 'intercase_n_3__Receive sample state_Callback timeout_Send notification',
 'intercase_n_3__Receive sample state_Export result_Export to EMS',
 'intercase_n_3__Receive sample state_Export result_Send notification',
 'intercase_n_3__Receive sample state_Export to EMS_Export result',
 'intercase_n_3__Receive sample state_Export to EMS_Send notification',
 'intercase_n_3__Receive sample state_Send notification_Callback timeout',
 'intercase_n_3__Receive sample state_Send notification_Export result',
 'intercase_n_3__Receive sample state_Send notification_Export to EMS',
 'intercase_n_3__Send notification_Export result_Export to EMS',
 'intercase_n_3__Send notification_Export to EMS_Export result',
 'intercase_n_3__Send notification_Receive sample state_Callback timeout',
 'intercase_n_3__Send notification_Receive sample state_Export result',
 'intercase_n_3__Send notification_Receive sample state_Export to EMS',
 'intercase_n_3__Wait for plate validation',
 'intercase_n_3__Wait for plate validation_Match patient data',
 'intercase_n_3__Wait for plate validation_Match patient data_Receive sample state',
 'intercase_n_3__Wait for plate validation_Match patient data_Send notification',
 'intercase_n_3__Wait for plate validation_Match patient data_timeout',
 'intercase_n_3__Wait for plate validation_Receive sample state',
 'intercase_n_3__Wait for plate validation_Receive sample state_Callback timeout',
 'intercase_n_3__Wait for plate validation_Receive sample state_Export result',
 'intercase_n_3__Wait for plate validation_Receive sample state_Export to EMS',
 'intercase_n_3__Wait for plate validation_Receive sample state_Send notification',
 'intercase_n_3__Wait for plate validation_Send notification_Receive sample state',
 'intercase_n_3__Wait for plate validation_timeout',
 'intercase_n_3__Wait for plate validation_timeout_Match patient data',
 'intercase_n_3__Wait for plate validation_timeout_Receive sample state',
 'intercase_n_3__Wait for plate validation_timeout_Send notification',
 'intercase_n_3__timeout',
 'intercase_n_3__timeout_Match patient data',
 'intercase_n_3__timeout_Match patient data_Match patient data',
 'intercase_n_3__timeout_Match patient data_Receive sample state',
 'intercase_n_3__timeout_Match patient data_Send notification',
 'intercase_n_3__timeout_Match patient data_Wait for plate validation',
 'intercase_n_3__timeout_Receive sample state_Callback timeout',
 'intercase_n_3__timeout_Receive sample state_Export result',
 'intercase_n_3__timeout_Receive sample state_Export to EMS',
 'intercase_n_3__timeout_Receive sample state_Send notification',
 'intercase_n_3__timeout_Send notification_Receive sample state',
 'intercase_n_3__timeout_Wait for plate validation',
 'intercase_n_3__timeout_Wait for plate validation_Match patient data',
 'intercase_n_3__timeout_Wait for plate validation_Receive sample state',
 'intercase_n_3__timeout_Wait for plate validation_Send notification'
]

In [4]:
n_processes = 32
batch_size = 16
N = 1000

In [5]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['A'], SampleOutcomes_PCR_QuantileRegression_A, {
                                                        'case_id_key' : 'case:concept:name',
                                                        'activity_key' : 'concept:name',
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

In [6]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-23.44251588263338338565266110')

In [7]:
np.mean(get_pscores(likelihoods_A))

np.float64(14858.38029841889)

In [8]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['AS'], SampleOutcomes_PCR_QuantileRegression_AS, {
                                                        'case_id_key' : 'case:concept:name',
                                                        'activity_key' : 'concept:name',
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

In [9]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-10540.65030097079831503029694')

In [10]:
np.mean(get_pscores(likelihoods_A))

np.float64(16032.120716800566)

In [11]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ASAC'], SampleOutcomes_PCR_QuantileRegression_ASAC, {
                                                        'case_id_key' : 'case:concept:name',
                                                        'activity_key' : 'concept:name',
                                                        'known_activities' : activity_count
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

In [12]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-Infinity')

In [13]:
np.mean(get_pscores(likelihoods_A))

np.float64(20718.070436875838)

In [14]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ASD'], SampleOutcomes_PCR_QuantileRegression_ASD, {
                                                        'case_id_key' : 'case:concept:name',
                                                        'activity_key' : 'concept:name',
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

In [15]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-Infinity')

In [16]:
np.mean(get_pscores(likelihoods_A))

np.float64(14437.950280440453)

In [17]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ASDII1'], SampleOutcomes_PCR_QuantileRegression_ASDII, {
                                                        'case_id_key' : 'case:concept:name',
                                                        'activity_key' : 'concept:name',
                                                        'known_activities' : activity_count,
                                                        'inter_instance_column_names' : ii1
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

In [18]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-Infinity')

In [19]:
np.mean(get_pscores(likelihoods_A))

np.float64(11845.16638971369)

In [20]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ASDII3'], SampleOutcomes_PCR_QuantileRegression_ASDII, {
                                                        'case_id_key' : 'case:concept:name',
                                                        'activity_key' : 'concept:name',
                                                        'known_activities' : activity_count,
                                                        'inter_instance_column_names' : ii3
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

In [21]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-Infinity')

In [22]:
np.mean(get_pscores(likelihoods_A))

np.float64(13360.89527057759)

In [23]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ASDACII1'], SampleOutcomes_PCR_QuantileRegression_ASDACII, {
                                                        'case_id_key' : 'case:concept:name',
                                                        'activity_key' : 'concept:name',
                                                        'known_activities' : activity_count,
                                                        'inter_instance_column_names' : ii1
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

NameError: name 'SampleOutcomes_PCR_QuantileRegression_ASDACII' is not defined

In [24]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-Infinity')

In [25]:
np.mean(get_pscores(likelihoods_A))

np.float64(13360.89527057759)

In [26]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ASDACII3'], SampleOutcomes_PCR_QuantileRegression_ASDACII, {
                                                        'case_id_key' : 'case:concept:name',
                                                        'activity_key' : 'concept:name',
                                                        'known_activities' : activity_count,
                                                        'inter_instance_column_names' : ii3
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

NameError: name 'SampleOutcomes_PCR_QuantileRegression_ASDACII' is not defined

In [27]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-Infinity')

In [28]:
np.mean(get_pscores(likelihoods_A))

np.float64(13360.89527057759)